In [1]:
import os
import json
import re
import sys
from pathlib import Path
from datetime import datetime
from typing import Optional, TypedDict, Annotated
import operator

ROOT = Path(r"C:\Users\MAITHILI\Care_Companion")
sys.path.insert(0, str(ROOT))

os.environ["GROQ_API_KEY"] = "************"
os.environ["USDA_API_KEY"] = "********"

from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

print("Imports done.")

Imports done.


In [2]:
#SQLite
import json
from sqlalchemy import (
    create_engine, Column, String,
    Integer, Float, Text
)
from sqlalchemy.orm import declarative_base, Session
from typing import Optional

Base      = declarative_base()
DB_PATH   = ROOT / "memory" / "sqlite" / "carecompanion.db"

class Patient(Base):
    __tablename__    = "patients"
    patient_id       = Column(String,  primary_key=True)
    name             = Column(String)
    age              = Column(Integer)
    created_at       = Column(String)
    last_seen        = Column(String)
    conditions       = Column(Text)
    medications      = Column(Text)
    allergies        = Column(Text)
    recent_glucose   = Column(Float)
    recent_a1c       = Column(Float)
    recent_bmi       = Column(Float)
    diet_preferences = Column(Text)
    foods_to_avoid   = Column(Text)
    activity_level   = Column(String)
    budget_monthly   = Column(Float)
    insurance        = Column(String)
    session_count    = Column(Integer, default=0)
    agent_notes      = Column(Text)
    last_fda_check   = Column(String, nullable=True)

engine = create_engine(f"sqlite:///{DB_PATH}", echo=False)
Base.metadata.create_all(engine)

def load_patient(patient_id: str) -> Optional[dict]:
    with Session(engine) as s:
        row = s.get(Patient, patient_id)
        if row is None:
            return None
        return {
            "patient_id":       row.patient_id,
            "name":             row.name,
            "age":              row.age,
            "created_at":       row.created_at,
            "last_seen":        row.last_seen,
            "conditions":       json.loads(row.conditions),
            "medications":      json.loads(row.medications),
            "allergies":        json.loads(row.allergies),
            "recent_glucose":   row.recent_glucose,
            "recent_a1c":       row.recent_a1c,
            "recent_bmi":       row.recent_bmi,
            "diet_preferences": json.loads(row.diet_preferences),
            "foods_to_avoid":   json.loads(row.foods_to_avoid),
            "activity_level":   row.activity_level,
            "budget_monthly":   row.budget_monthly,
            "insurance":        row.insurance,
            "session_count":    row.session_count,
            "agent_notes":      json.loads(row.agent_notes),
            "last_fda_check":   row.last_fda_check,
        }

def update_patient_field(patient_id: str, field: str, value):
    with Session(engine) as s:
        row = s.get(Patient, patient_id)
        if row:
            setattr(row, field,
                    json.dumps(value) if isinstance(value, (list, dict))
                    else value)
            row.last_seen = datetime.now().isoformat()
            s.commit()

print("SQLite ready.")

#ChromaDB
import chromadb

chroma_client = chromadb.PersistentClient(
    path=str(ROOT / "memory" / "chroma_db")
)
conversations = chroma_client.get_or_create_collection(
    name="conversations",
    metadata={"hnsw:space": "cosine"}
)

def save_message(patient_id, role, content, session_id):
    doc_id = f"{patient_id}_{session_id}_{datetime.now().timestamp()}"
    conversations.add(
        documents=[content],
        metadatas=[{"patient_id": patient_id, "role": role,
                    "session_id": session_id,
                    "timestamp":  datetime.now().isoformat()}],
        ids=[doc_id]
    )

def get_relevant_history(patient_id, query, n=3):
    count = conversations.count()
    if count == 0:
        return []
    results = conversations.query(
        query_texts=[query],
        n_results=min(n, count),
        where={"patient_id": patient_id}
    )
    history = []
    for doc, meta in zip(results["documents"][0],
                         results["metadatas"][0]):
        history.append({
            "role":    meta["role"],
            "content": doc,
            "session": meta["session_id"]
        })
    return history

print("ChromaDB ready.")

#Tools
import requests
from tenacity import retry, stop_after_attempt, wait_exponential

@retry(stop=stop_after_attempt(3),
       wait=wait_exponential(min=1, max=5))
def check_drug_interactions(medications):
    results = {}
    for med in medications:
        drug_name = med["name"] if isinstance(med, dict) else med
        try:
            resp = requests.get(
                "https://api.fda.gov/drug/label.json",
                params={"search": f'openfda.generic_name:"{drug_name.lower()}"',
                        "limit": 1},
                timeout=10
            )
            if resp.status_code != 200:
                results[drug_name] = {"status": "not_found"}
                continue
            label = resp.json()["results"][0]
            results[drug_name] = {
                "status":            "found",
                "boxed_warning":     label.get("boxed_warning",     ["None"])[0][:300],
                "drug_interactions": label.get("drug_interactions", ["None"])[0][:400],
                "warnings":          label.get("warnings",          ["None"])[0][:300],
            }
        except Exception as e:
            results[drug_name] = {"status": "error", "error": str(e)}
    return results

def get_medication_cost(drug_name):
    PRICES = {
        "metformin":        {"brand": "Glucophage",  "brand_price": 45,  "generic_price": 4},
        "glipizide":        {"brand": "Glucotrol",   "brand_price": 38,  "generic_price": 9},
        "sitagliptin":      {"brand": "Januvia",     "brand_price": 550, "generic_price": 45},
        "lisinopril":       {"brand": "Zestril",     "brand_price": 65,  "generic_price": 8},
        "insulin glargine": {"brand": "Lantus",      "brand_price": 340, "generic_price": 98},
    }
    key   = drug_name.lower()
    info  = PRICES.get(key)
    if info:
        savings = info["brand_price"] - info["generic_price"]
        return {
            "drug_name":       drug_name,
            "brand_name":      info["brand"],
            "generic_price":   f"${info['generic_price']}/month",
            "annual_savings":  f"${savings * 12}",
            "recommendation":  (f"Ask for generic {drug_name} instead of "
                                f"{info['brand']}. Save ${savings}/month.")
        }
    return {"drug_name": drug_name,
            "recommendation": f"Ask your pharmacist about generics for {drug_name}."}

def lookup_nutrition(food_name):
    try:
        resp = requests.get(
            "https://api.nal.usda.gov/fdc/v1/foods/search",
            params={"query": food_name, "pageSize": 1,
                    "api_key": os.environ.get("USDA_API_KEY", "DEMO_KEY")},
            timeout=15
        )
        foods = resp.json().get("foods", [])
        if not foods:
            return {"food": food_name, "status": "not_found"}
        food = foods[0]
        nraw = {n["nutrientName"]: n["value"]
                for n in food.get("foodNutrients", [])}
        carbs     = nraw.get("Carbohydrate, by difference", 0)
        fiber     = nraw.get("Fiber, total dietary", 0)
        net_carbs = max(0, carbs - fiber)
        if net_carbs < 5:
            flag = "green";  assessment = "Excellent for blood sugar control"
        elif net_carbs < 15:
            flag = "yellow"; assessment = "Moderate — eat in controlled portions"
        elif net_carbs < 30:
            flag = "orange"; assessment = "Caution — limit portion size"
        else:
            flag = "red";    assessment = "Avoid — will spike blood sugar"
        return {"food": food["description"], "net_carbs_g": round(net_carbs,1),
                "flag": flag, "assessment": assessment, "status": "found"}
    except Exception as e:
        return {"food": food_name, "status": "error", "error": str(e)}

def get_fda_alerts(medications):
    alerts = {}
    for med in medications:
        drug_name   = med["name"] if isinstance(med, dict) else med
        drug_alerts = []
        try:
            resp = requests.get(
                "https://api.fda.gov/drug/enforcement.json",
                params={"search": f'product_description:"{drug_name}"',
                        "limit": 2},
                timeout=10
            )
            if resp.status_code == 200:
                for r in resp.json().get("results", []):
                    drug_alerts.append({
                        "type":        "RECALL",
                        "description": r.get("reason_for_recall", "")[:150],
                        "date":        r.get("recall_initiation_date", ""),
                    })
        except Exception:
            pass
        alerts[drug_name] = {
            "alerts_found": len(drug_alerts),
            "alerts":       drug_alerts,
            "summary": ("No active alerts." if not drug_alerts
                        else f"{len(drug_alerts)} alert(s) found — review recommended.")
        }
    return alerts

def search_pubmed(query, max_results=2):
    try:
        ids = requests.get(
            "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi",
            params={"db": "pubmed", "term": query,
                    "retmax": max_results, "retmode": "json"},
            timeout=10
        ).json()["esearchresult"]["idlist"]
        if not ids:
            return []
        summaries = requests.get(
            "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi",
            params={"db": "pubmed", "id": ",".join(ids), "retmode": "json"},
            timeout=10
        ).json().get("result", {})
        return [{"pmid":    pmid,
                 "title":   summaries.get(pmid, {}).get("title", ""),
                 "year":    summaries.get(pmid, {}).get("pubdate", "")[:4],
                 "url":     f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/"}
                for pmid in ids]
    except Exception:
        return []

print("All tools ready.")

SQLite ready.
ChromaDB ready.
All tools ready.


In [3]:
class MemoryManager:
    def __init__(self, patient_id: str):
        self.patient_id = patient_id
        self.session_id = f"session_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        self.profile    = load_patient(patient_id)
        if self.profile is None:
            raise ValueError(f"Patient {patient_id} not found.")
        self.profile["session_count"] += 1
        update_patient_field(patient_id, "session_count",
                             self.profile["session_count"])

    def remember(self, role: str, content: str):
        save_message(self.patient_id, role, content, self.session_id)

    def recall(self, query: str, n: int = 3) -> list:
        return get_relevant_history(self.patient_id, query, n)

    def get_profile(self) -> dict:
        return load_patient(self.patient_id)

    def update_profile(self, field: str, value):
        update_patient_field(self.patient_id, field, value)

    def add_note(self, note: str):
        profile = self.get_profile()
        notes   = profile["agent_notes"]
        notes.append(f"[{datetime.now().strftime('%Y-%m-%d')}] {note}")
        self.update_profile("agent_notes", notes)

    def get_context_summary(self) -> str:
        p    = self.get_profile()
        meds = ", ".join(
            f"{m['name']} {m['dose']} {m['frequency']}"
            for m in p["medications"]
        )
        notes = ("\n".join(p["agent_notes"][-3:])
                 if p["agent_notes"] else "None yet")
        return f"""PATIENT PROFILE:
- Name: {p['name']}, Age: {p['age']}
- Conditions: {', '.join(p['conditions'])}
- Medications: {meds}
- Recent Glucose: {p['recent_glucose']} mg/dL
- Recent A1c: {p['recent_a1c']}%
- BMI: {p['recent_bmi']}
- Diet preferences: {', '.join(p['diet_preferences'])}
- Activity level: {p['activity_level']}
- Monthly budget: ${p['budget_monthly']}
- Insurance: {p['insurance']}
- Sessions so far: {p['session_count']}
- Agent notes:
{notes}"""

print("MemoryManager ready.")

MemoryManager ready.


In [4]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.2,
    api_key=os.environ["GROQ_API_KEY"]
)

SYSTEM_PROMPT = """You are CareCompanion AI, a knowledgeable and empathetic 
health assistant for patients managing Type 2 Diabetes.

You have access to the patient's full medical profile and conversation history.
You can call tools to check drug interactions, find cheaper medications, 
look up nutrition info, check FDA alerts, and search medical research.

Important rules:
- Always personalize responses using the patient's profile
- When discussing food, always check nutrition data
- When discussing medications, always check costs and interactions
- Be warm and supportive, not clinical and cold
- Always remind the patient you are not a replacement for their doctor
- Keep responses clear and easy to understand

You will decide which tools to call based on what the patient asks."""

print(f"Model: llama-3.3-70b-versatile")

Model: llama-3.3-70b-versatile


In [5]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator

class AgentState(TypedDict):
    """
    The state that flows through every node in the agent graph.
    Every node reads from this and writes back to it.
    """
    patient_id:   str
    user_message: str
    profile:      dict
    history:      list       
    tool_results: dict       
    tools_used:   list       
    response:     str        
    session_id:   str

print("AgentState defined.")
print("Fields: patient_id, user_message, profile, history,tool_results, tools_used, response, session_id")

AgentState defined.
Fields: patient_id, user_message, profile, history,tool_results, tools_used, response, session_id


In [7]:
def node_load_context(state: AgentState) -> AgentState:
    """
    Node 1: Load patient profile and relevant memory.
    Runs first on every message.
    """
    memory  = MemoryManager(state["patient_id"])
    profile = memory.get_profile()

    #retrieving semantically relevant past messages
    history = memory.recall(state["user_message"], n=4)

    state["profile"]    = profile
    state["history"]    = history
    state["session_id"] = memory.session_id

    print(f"[Node 1] Context loaded for {profile['name']}")
    print(f"         Recalled {len(history)} relevant past messages")
    return state


def node_decide_tools(state: AgentState) -> AgentState:
    """
    Node 2: LLM decides which tools to call based on the message.
    Returns a list of tool names to execute.
    """
    profile = state["profile"]
    msg     = state["user_message"].lower()

    #Building decision prompt
    decision_prompt = f"""
You are deciding which tools to call for a patient query.

Patient: {profile['name']}, {profile['age']}y, {', '.join(profile['conditions'])}
Medications: {[m['name'] for m in profile['medications']]}
User message: "{state['user_message']}"

Available tools:
- check_drug_interactions: use when patient asks about medications, side effects, safety
- get_medication_cost: use when patient mentions cost, price, affordability, insurance
- lookup_nutrition: use when patient mentions any food, diet, eating, meal
- get_fda_alerts: use at session start OR when patient asks about drug safety
- search_pubmed: use when patient asks about research, evidence, studies

Reply with ONLY a JSON list of tool names to call. Example: ["lookup_nutrition", "search_pubmed"]
If no tools needed, reply: []
"""
    response = llm.invoke([HumanMessage(content=decision_prompt)])


    try:
        
        match = re.search(r'\[.*?\]', response.content, re.DOTALL)
        tools_to_use = json.loads(match.group()) if match else []
    except Exception:
        tools_to_use = []


    if profile["session_count"] <= 1 and "get_fda_alerts" not in tools_to_use:
        tools_to_use.append("get_fda_alerts")

    state["tools_used"] = tools_to_use
    print(f"[Node 2] Tools selected: {tools_to_use}")
    return state


def node_run_tools(state: AgentState) -> AgentState:
    """
    Node 3: Execute the selected tools and collect results.
    """
    profile      = state["profile"]
    meds         = profile["medications"]
    tool_results = {}

    for tool_name in state["tools_used"]:
        print(f"[Node 3] Running: {tool_name}")
        try:
            if tool_name == "check_drug_interactions":
                tool_results["interactions"] = check_drug_interactions(meds)

            elif tool_name == "get_medication_cost":
                tool_results["costs"] = {
                    m["name"]: get_medication_cost(m["name"])
                    for m in meds
                }

            elif tool_name == "lookup_nutrition":
                
                food_keywords = ["rice", "bread", "sugar", "fruit", "vegetable",
                                 "oatmeal", "pasta", "potato", "juice", "milk",
                                 "meat", "chicken", "fish", "egg", "cheese"]
                msg_lower = state["user_message"].lower()
                foods_mentioned = [f for f in food_keywords if f in msg_lower]
                if not foods_mentioned:
                    foods_mentioned = ["the food mentioned"]
                tool_results["nutrition"] = {
                    food: lookup_nutrition(food)
                    for food in foods_mentioned[:2]
                }

            elif tool_name == "get_fda_alerts":
                tool_results["fda_alerts"] = get_fda_alerts(meds)

            elif tool_name == "search_pubmed":
                query = f"{state['user_message']} type 2 diabetes"
                tool_results["research"] = search_pubmed(query, max_results=2)

        except Exception as e:
            tool_results[tool_name] = {"error": str(e)}

    state["tool_results"] = tool_results
    return state


def node_generate_response(state: AgentState) -> AgentState:
    """
    Node 4: Generate the final personalized response
    using profile + memory + tool results.
    """
    profile = state["profile"]
    memory  = MemoryManager(state["patient_id"])

    history_text = ""
    if state["history"]:
        history_text = "\n".join(
            f"  [{m['role'].upper()}]: {m['content'][:100]}"
            for m in state["history"]
        )
    else:
        history_text = "  No previous conversations yet."


    tool_text = ""
    if state["tool_results"]:
        tool_text = json.dumps(state["tool_results"], indent=2)[:1500]
    else:
        tool_text = "No tools called for this message."

    #final generation prompt
    final_prompt = f"""
{SYSTEM_PROMPT}

{memory.get_context_summary()}

RELEVANT PAST CONVERSATIONS:
{history_text}

TOOL RESULTS:
{tool_text}

PATIENT SAYS: "{state['user_message']}"

Write a warm, personalized, helpful response using all the above context.
Reference specific details from the patient's profile.
If tool results contain useful data, include it naturally in your response.
End with one follow-up question to learn more about the patient.
"""
    response = llm.invoke([HumanMessage(content=final_prompt)])

    #saving to memory
    memory.remember("user",      state["user_message"])
    memory.remember("assistant", response.content)

    note_prompt = f"""
In one short sentence, what is the most important thing to remember 
about this patient interaction?
Patient said: "{state['user_message']}"
Agent responded about: {list(state['tool_results'].keys())}
Reply with ONLY the one sentence note, no preamble.
"""
    note_resp = llm.invoke([HumanMessage(content=note_prompt)])
    memory.add_note(note_resp.content.strip())

    state["response"] = response.content
    print(f"[Node 4] Response generated ({len(response.content)} chars)")
    return state

print("All 4 agent nodes defined.")

All 4 agent nodes defined.


In [8]:
#wiring the nodes into a graph
graph = StateGraph(AgentState)

#adding nodes
graph.add_node("load_context",       node_load_context)
graph.add_node("decide_tools",       node_decide_tools)
graph.add_node("run_tools",          node_run_tools)
graph.add_node("generate_response",  node_generate_response)

#linear flow
graph.set_entry_point("load_context")
graph.add_edge("load_context",      "decide_tools")
graph.add_edge("decide_tools",      "run_tools")
graph.add_edge("run_tools",         "generate_response")
graph.add_edge("generate_response", END)

agent = graph.compile()

print("LangGraph agent compiled.")
print()
print("Flow: load_context -> decide_tools -> run_tools -> generate_response -> END")

LangGraph agent compiled.

Flow: load_context -> decide_tools -> run_tools -> generate_response -> END


In [9]:
def chat(patient_id: str, message: str) -> str:
    """
    Single entry point for talking to the agent.
    Pass a patient ID and a message, get a response back.
    """
    print(f"\n{'='*55}")
    print(f"USER: {message}")
    print(f"{'='*55}")

    result = agent.invoke({
        "patient_id":   patient_id,
        "user_message": message,
        "profile":      {},
        "history":      [],
        "tool_results": {},
        "tools_used":   [],
        "response":     "",
        "session_id":   "",
    })

    print(f"\nAGENT: {result['response']}")
    print(f"\nTools used: {result['tools_used']}")
    return result["response"]

print("chat() function ready.")
print("Usage: chat('p001', 'your message here')")

chat() function ready.
Usage: chat('p001', 'your message here')


In [10]:
#testing the full agent

#full conversation test with Sarah across 3 messages
print("CARECOMPANION AI — AGENT TEST")
print("Patient: Sarah (p001)")
print("=" * 55)

# Message 1: Medication cost question
r1 = chat("p001", "I'm worried about the cost of my medications, they're getting expensive")

print("\n" + "-"*55)

# Message 2: Food question  
r2 = chat("p001", "Is oatmeal okay for breakfast? I've been eating it every morning")

print("\n" + "-"*55)

# Message 3: General health question
r3 = chat("p001", "My glucose was 132 this week, down from 148. Am I making progress?")

CARECOMPANION AI — AGENT TEST
Patient: Sarah (p001)

USER: I'm worried about the cost of my medications, they're getting expensive
[Node 1] Context loaded for Sarah
         Recalled 4 relevant past messages
[Node 2] Tools selected: ['get_medication_cost']
[Node 3] Running: get_medication_cost
[Node 4] Response generated (1136 chars)

AGENT: Hi Sarah, I completely understand your concern about the cost of your medications. We've discussed this before, and I know how worried you are about staying within your monthly budget of $200. I've checked on the costs of your current medication, Metformin 500mg, which you're taking twice a day. I found out that you can actually save some money by asking for the generic version of Metformin instead of the brand name, Glucophage. By doing so, you could save around $41 per month, which translates to $492 per year. That's a significant amount, especially considering your budget.

I also want to remind you that you've been doing great with avoiding whi

In [11]:
#testing memory across sessions

#This to prove the agent remembers across separate sessions
print("MEMORY TEST — New session, same patient")
print("The agent should remember things from the previous session")
print("=" * 55)

# This is a new session, agent has NO current conversation context, but ChromaDB has the past messages stored
r4 = chat("p001", "You mentioned something about rice before, can you remind me?")

MEMORY TEST — New session, same patient
The agent should remember things from the previous session

USER: You mentioned something about rice before, can you remind me?
[Node 1] Context loaded for Sarah
         Recalled 4 relevant past messages
[Node 2] Tools selected: ['lookup_nutrition']
[Node 3] Running: lookup_nutrition
[Node 4] Response generated (1422 chars)

AGENT: Hi Sarah, I'm so glad you're thinking about your food choices and how they impact your Type 2 Diabetes management. I remember we discussed rice before, and I'm happy to remind you about it. As you know, you've been doing a great job avoiding white rice, which is a good idea since it's high in carbs. According to our nutrition data, one serving of rice contains about 25 grams of net carbs, which is why we've flagged it as "orange" - meaning you should limit your portion size. 

I'm proud of you for making conscious decisions about your diet, especially since you've been consistently eating oatmeal for breakfast. I've b

In [12]:
#proactive fda monitor

def run_proactive_monitor(patient_ids: list) -> dict:
    """
    Runs at session start automatically.
    Checks all patients' medications for new FDA alerts
    and flags anything that needs attention.
    
    In a real deployment this would run on a schedule.
    """
    print("Running proactive FDA monitor...")
    print("=" * 55)
    
    flagged = {}

    for pid in patient_ids:
        profile = load_patient(pid)
        if not profile:
            continue

        alerts  = get_fda_alerts(profile["medications"])
        has_alerts = any(
            info["alerts_found"] > 0
            for info in alerts.values()
        )

        if has_alerts:
            flagged[pid] = {
                "name":    profile["name"],
                "alerts":  alerts
            }
            print(f"  FLAGGED: {profile['name']} ({pid})")
            for drug, info in alerts.items():
                if info["alerts_found"] > 0:
                    print(f"    {drug}: {info['summary']}")
        else:
            print(f"  OK:      {profile['name']} ({pid}) — no alerts")

    print(f"\nMonitor complete. {len(flagged)} patient(s) flagged.")
    return flagged


#Running on first 5 test patients
flagged_patients = run_proactive_monitor(
    [f"p{i:03d}" for i in range(1, 6)]
)

Running proactive FDA monitor...
  FLAGGED: Sarah (p001)
    Metformin: 2 alert(s) found — review recommended.
  FLAGGED: James (p002)
    Metformin: 2 alert(s) found — review recommended.
    Glipizide: 2 alert(s) found — review recommended.
  FLAGGED: Maria (p003)
    Metformin: 2 alert(s) found — review recommended.
    Lisinopril: 2 alert(s) found — review recommended.
  FLAGGED: David (p004)
    Insulin Glargine: 2 alert(s) found — review recommended.
  FLAGGED: Linda (p005)
    Sitagliptin: 2 alert(s) found — review recommended.

Monitor complete. 5 patient(s) flagged.
